# 🦞 OpenClaw × ArangoDB — Digital Brain

**Replace OpenClaw's SQLite memory layer with ArangoDB as a unified graph + vector + document brain.**

OpenClaw's default memory stack:
- `MEMORY.md` / `memory/YYYY-MM-DD.md` → flat Markdown files
- SQLite (FTS5 + sqlite-vec) → keyword + semantic search
- Optional QMD sidecar for GGUF embeddings

What we're building:
- **ArangoDB** replaces SQLite entirely
- **Document store** → memory entries (facts, events, notes, conversations)
- **Graph layer** → entity linking, causal chains, temporal DAG
- **Vector index** → semantic `memory_search` via ArangoDB's built-in vector search
- **`memory_search` / `memory_get` / `memory_store`** → drop-in tool replacements
- **Heartbeat sync** → periodic compaction into the graph brain

---
```
OpenClaw Gateway
     │
     ├── memory_store(fact)      → ArangoDB memories collection
     ├── memory_search(query)    → AQL + vector search
     ├── memory_get(path/key)    → document fetch by key
     └── entity_link(a, b, rel)  → edge insert in knowledge graph
```


## 1. Install Dependencies


In [2]:
!pip install -q python-arango sentence-transformers numpy python-dateutil rich


## 2. Connect to ArangoDB


In [2]:
from arango import ArangoClient
from rich.console import Console
from rich.table import Table

console = Console()

ARANGO_HOST = 'https://5ieeavs2.rnd.pilot.arango.ai'
ARANGO_USER = 'root'
ARANGO_PASS = 'U@$GTEaYL&TgO:pO'
DB_NAME     = 'openclaw_brain'

client = ArangoClient(hosts=ARANGO_HOST)
sys_db = client.db('_system', username=ARANGO_USER, password=ARANGO_PASS, verify=True)

if not sys_db.has_database(DB_NAME):
    sys_db.create_database(DB_NAME)
    console.print(f'[green]✓[/green] Created database: [bold]{DB_NAME}[/bold]')
else:
    console.print(f'[cyan]→[/cyan] Database already exists: [bold]{DB_NAME}[/bold]')

db = client.db(DB_NAME, username=ARANGO_USER, password=ARANGO_PASS, verify=True)
console.print(f'[green]✓[/green] Connected to [bold]{DB_NAME}[/bold] @ {ARANGO_HOST}')


→ Database already exists: openclaw_brain

✓ Connected to openclaw_brain @ https://5ieeavs2.rnd.pilot.arango.ai

## 3. Schema — Collections, Indexes & Graph


In [3]:
# Collections:
#  memories     - individual memory entries (facts, notes, events, messages)
#  entities     - named entities extracted from memories
#  memory_edges - temporal / causal / summary edges between memories
#  entity_edges - typed relations between entities
#  sessions     - conversation session metadata
#  daily_logs   - per-day compacted summaries

VERTEX_COLS = ['memories', 'entities', 'sessions', 'daily_logs']
EDGE_COLS   = ['memory_edges', 'entity_edges']
GRAPH_NAME  = 'brain_graph'

def ensure_collection(name, edge=False):
    if not db.has_collection(name):
        db.create_collection(name, edge=edge)
        console.print(f'  [green]+[/green] Created {"edge" if edge else "vertex"} col: {name}')
    else:
        console.print(f'  [dim]·[/dim] Already exists: {name}')

console.print('[bold]Setting up collections...[/bold]')
for col in VERTEX_COLS: ensure_collection(col)
for col in EDGE_COLS:   ensure_collection(col, edge=True)

if not db.has_graph(GRAPH_NAME):
    db.create_graph(
        GRAPH_NAME,
        edge_definitions=[
            {'edge_collection': 'memory_edges',
             'from_vertex_collections': ['memories', 'daily_logs', 'sessions'],
             'to_vertex_collections':   ['memories', 'daily_logs', 'sessions']},
            {'edge_collection': 'entity_edges',
             'from_vertex_collections': ['entities'],
             'to_vertex_collections':   ['entities', 'memories']},
        ]
    )
    console.print(f'[green]✓[/green] Created graph: [bold]{GRAPH_NAME}[/bold]')
else:
    console.print(f'[cyan]→[/cyan] Graph already exists: [bold]{GRAPH_NAME}[/bold]')

console.print('[bold]Setting up indexes...[/bold]')
mem_col = db.collection('memories')

try:
    mem_col.add_fulltext_index(fields=['content'])
    console.print('  [green]+[/green] Fulltext index on memories.content')
except Exception as e:
    console.print(f'  [yellow]·[/yellow] Fulltext index skipped: {e}')

try:
    mem_col.add_persistent_index(fields=['memory_type', 'created_at'])
    console.print('  [green]+[/green] Persistent index on memory_type + created_at')
except Exception as e:
    console.print(f'  [yellow]·[/yellow] Persistent index skipped: {e}')

try:
    mem_col.add_ttl_index(fields=['expires_at'], expiry_time=0)
    console.print('  [green]+[/green] TTL index on expires_at')
except Exception as e:
    console.print(f'  [yellow]·[/yellow] TTL index skipped: {e}')

try:
    db.collection('entities').add_persistent_index(
        fields=['name', 'entity_type'], unique=True, sparse=True)
    console.print('  [green]+[/green] Unique index on entities.name+entity_type')
except Exception as e:
    console.print(f'  [yellow]·[/yellow] Entity index skipped: {e}')

console.print('[bold green]✓ Schema ready[/bold green]')


Setting up collections...

· Already exists: memories

· Already exists: entities

· Already exists: sessions

· Already exists: daily_logs

...

/tmp/ipykernel_1427/1397447556.py:56: DeprecationWarning: add_ttl_index is deprecated. Using add_index with {'type': 'ttl'} instead.
  mem_col.add_ttl_index(fields=['expires_at'], expiry_time=0)


+ TTL index on expires_at

/tmp/ipykernel_1427/1397447556.py:62: DeprecationWarning: add_persistent_index is deprecated. Using add_index with {'type': 'persistent'} instead.
  db.collection('entities').add_persistent_index(


+ Unique index on entities.name+entity_type

✓ Schema ready

## 4. Embedding Model & Vector Index


In [4]:
from sentence_transformers import SentenceTransformer

console.print('[bold]Loading embedding model...[/bold]')
EMBED_MODEL = SentenceTransformer('all-MiniLM-L6-v2')
EMBED_DIM   = 384
console.print(f'[green]✓[/green] Model loaded: all-MiniLM-L6-v2 ({EMBED_DIM}d)')

def embed(text: str) -> list:
    return EMBED_MODEL.encode(text, normalize_embeddings=True).tolist()

# Try native ArangoDB vector index (3.12+)
VECTOR_INDEX_NATIVE = False
try:
    mem_col.add_index({
        'type': 'vector',
        'fields': ['embedding'],
        'params': {'metric': 'cosine', 'dimension': EMBED_DIM, 'nLists': 2}
    })
    VECTOR_INDEX_NATIVE = True
    console.print('[green]✓[/green] Native vector index created on memories.embedding')
except Exception as e:
    console.print(f'[yellow]⚠[/yellow] Native vector index not available ({e.__class__.__name__}). '
                  'Falling back to AQL cosine similarity.')


Loading embedding model...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded: all-MiniLM-L6-v2 (384d)

✓ Native vector index created on memories.embedding

## 5. The Digital Brain

| OpenClaw tool | Brain method |
|---|---|
| `memory_store(content)` | `brain.store(...)` |
| `memory_search(query)` | `brain.search(...)` |
| `memory_get(key)` | `brain.get(key)` |
| entity linking | `brain.link_entities(a, b, rel)` |
| heartbeat compaction | `brain.compact_day(date)` |


In [5]:
from datetime import datetime, timezone, date
import hashlib, re
from typing import Optional

class DigitalBrain:
    """ArangoDB-backed memory layer for OpenClaw agents."""

    MEMORY_TYPES = {'fact','event','note','conversation','decision','preference','daily_summary'}

    def __init__(self, db, embed_fn, vector_native=False):
        self.db          = db
        self.embed       = embed_fn
        self.native_vec  = vector_native
        self.memories    = db.collection('memories')
        self.entities    = db.collection('entities')
        self.mem_edges   = db.collection('memory_edges')
        self.ent_edges   = db.collection('entity_edges')
        self.sessions    = db.collection('sessions')
        self.daily_logs  = db.collection('daily_logs')

    # ── Store ──────────────────────────────────────────────────────────────
    def store(self, content, memory_type='fact', tags=None, source='agent',
              session_id=None, agent_id='default', expires_at=None,
              confidence=1.0, entities_mentioned=None):
        if memory_type not in self.MEMORY_TYPES: memory_type = 'note'
        now = datetime.now(timezone.utc).isoformat()
        key = hashlib.sha256(f'{agent_id}:{content}'.encode()).hexdigest()[:16]
        doc = {
            '_key': key, 'content': content, 'memory_type': memory_type,
            'tags': tags or [], 'source': source, 'session_id': session_id,
            'agent_id': agent_id, 'created_at': now, 'confidence': confidence,
            'access_count': 0, 'last_accessed': now,
            'embedding': self.embed(content), 'expires_at': expires_at,
        }
        result = self.memories.insert(doc, overwrite=True, return_new=True)['new']
        if entities_mentioned:
            for name in entities_mentioned:
                ent = self._upsert_entity(name)
                self._safe_edge({'_from': ent['_id'], '_to': result['_id'],
                                 'relation': 'mentioned_in', 'created_at': now}, self.ent_edges)
        return result

    # ── Search ─────────────────────────────────────────────────────────────
    def search(self, query, top_k=6, memory_type=None, agent_id='default'):
        q_vec = self.embed(query)
        type_filter = 'FILTER m.memory_type == @mtype' if memory_type else ''
        aql = f'''
        LET q = @q_vec
        FOR m IN memories
          FILTER m.agent_id == @agent_id
          FILTER m.embedding != null
          {type_filter}
          LET score = (
            LET pairs = (FOR i IN 0..LENGTH(m.embedding)-1 RETURN m.embedding[i] * q[i])
            RETURN SUM(pairs)
          )[0]
          SORT score DESC
          LIMIT @k
          RETURN MERGE(m, {{score: score}})
        '''
        bv = {'q_vec': q_vec, 'k': top_k, 'agent_id': agent_id}
        if memory_type: bv['mtype'] = memory_type
        results = list(self.db.aql.execute(aql, bind_vars=bv))
        for r in results:
            try:
                self.memories.update({'_key': r['_key'],
                    'access_count': r.get('access_count', 0) + 1,
                    'last_accessed': datetime.now(timezone.utc).isoformat()})
            except Exception: pass
        return results

    def get(self, key):
        try: return self.memories.get(key)
        except Exception: return None

    def delete(self, key):
        try: self.memories.delete(key); return True
        except Exception: return False

    # ── Entities ───────────────────────────────────────────────────────────
    def _upsert_entity(self, name, entity_type='unknown'):
        key = re.sub(r'[^a-zA-Z0-9_-]', '_', name.lower())[:64]
        doc = {'_key': key, 'name': name, 'entity_type': entity_type,
               'created_at': datetime.now(timezone.utc).isoformat()}
        try: return self.entities.insert(doc, return_new=True)['new']
        except Exception: return self.entities.get(key)

    def _safe_edge(self, doc, col):
        try: col.insert(doc)
        except Exception: pass

    def link_entities(self, a, b, relation, weight=1.0, bidirectional=False):
        ea = self._upsert_entity(a)
        eb = self._upsert_entity(b)
        now = datetime.now(timezone.utc).isoformat()
        edge = {'_from': ea['_id'], '_to': eb['_id'], 'relation': relation,
                'weight': weight, 'created_at': now}
        self._safe_edge(edge, self.ent_edges)
        if bidirectional:
            self._safe_edge({**edge, '_from': eb['_id'], '_to': ea['_id']}, self.ent_edges)
        console.print(f'  [cyan]↔[/cyan] {a} [bold]{relation}[/bold] {b}')

    # ── Sessions ───────────────────────────────────────────────────────────
    def open_session(self, session_id, agent_id='default', channel=None):
        now = datetime.now(timezone.utc).isoformat()
        doc = {'_key': session_id, 'agent_id': agent_id, 'channel': channel,
               'started_at': now, 'ended_at': None, 'message_count': 0}
        try: return self.sessions.insert(doc, return_new=True)['new']
        except Exception: return self.sessions.get(session_id)

    def store_message(self, session_id, role, content, agent_id='default'):
        mem = self.store(content, memory_type='conversation', source=role,
                         session_id=session_id, agent_id=agent_id)
        try:
            sess = self.sessions.get(session_id)
            if sess:
                self._safe_edge({'_from': f'sessions/{session_id}', '_to': mem['_id'],
                                 'relation': 'contains_message',
                                 'created_at': datetime.now(timezone.utc).isoformat()},
                                self.mem_edges)
                self.sessions.update({'_key': session_id,
                    'message_count': sess.get('message_count', 0) + 1})
        except Exception: pass
        return mem

    # ── Compaction ─────────────────────────────────────────────────────────
    def compact_day(self, target_date=None, agent_id='default'):
        target_date = target_date or date.today()
        day_str = target_date.isoformat()
        entries = list(self.db.aql.execute('''
            FOR m IN memories
              FILTER m.agent_id == @agent_id
              FILTER m.created_at >= @start AND m.created_at <= @end
              SORT m.created_at ASC
              RETURN {key: m._key, id: m._id, type: m.memory_type, content: m.content}
        ''', bind_vars={'agent_id': agent_id,
                        'start': f'{day_str}T00:00:00+00:00',
                        'end':   f'{day_str}T23:59:59+00:00'}))
        if not entries:
            console.print(f'[yellow]No memories to compact for {day_str}[/yellow]')
            return {}
        summary = f'# Daily Summary — {day_str}\n\n'
        by_type = {}
        for e in entries: by_type.setdefault(e['type'], []).append(e['content'])
        for t, cs in by_type.items():
            summary += f'## {t.title()}\n'
            for c in cs: summary += f'- {c}\n'
            summary += '\n'
        log_key = f'{agent_id}_{day_str}'
        log = {'_key': log_key, 'agent_id': agent_id, 'date': day_str,
               'entry_count': len(entries), 'summary': summary,
               'embedding': self.embed(summary[:512]),
               'created_at': datetime.now(timezone.utc).isoformat()}
        self.daily_logs.insert(log, overwrite=True)
        now = datetime.now(timezone.utc).isoformat()
        for e in entries:
            self._safe_edge({'_from': f'memories/{e["key"]}',
                             '_to': f'daily_logs/{log_key}',
                             'relation': 'compacted_into', 'created_at': now},
                            self.mem_edges)
        console.print(f'[green]✓[/green] Compacted {len(entries)} memories → daily log: {day_str}')
        return log

    # ── Graph traversal ────────────────────────────────────────────────────
    def entity_neighbourhood(self, entity_name, depth=2):
        key = re.sub(r'[^a-zA-Z0-9_-]', '_', entity_name.lower())[:64]
        try:
            return list(self.db.aql.execute('''
              FOR v, e, p IN 1..@depth ANY @start
                entity_edges, memory_edges
                OPTIONS {bfs: true, uniqueVertices: "global"}
                RETURN {vertex: v, edge_label: e.relation, depth: LENGTH(p.edges)}
            ''', bind_vars={'start': f'entities/{key}', 'depth': depth}))
        except Exception as ex:
            console.print(f'[red]Traversal error:[/red] {ex}'); return []

    def stats(self):
        return {k: db.collection(k).count()
                for k in ['memories','entities','memory_edges','entity_edges','sessions','daily_logs']}


brain = DigitalBrain(db, embed, vector_native=VECTOR_INDEX_NATIVE)
console.print('[bold green]✓ DigitalBrain ready[/bold green]')
console.print(brain.stats())


✓ DigitalBrain ready

{'memories': 144, 'entities': 9, 'memory_edges': 299, 'entity_edges': 44, 'sessions': 2, 'daily_logs': 1}

## 6. Seed — Enron Email Dataset

Downloads a sample of the public Enron email corpus and loads it as memories.
Emails become `conversation` memories; people become entities; org relationships become edges.
No personal data — this is purely the public Enron dataset used for graph/NLP research.


In [6]:
import os, re, hashlib

# ── Download Enron email sample from HuggingFace ────────────────────────────
# Uses the 'enron_spam' dataset - 30k+ emails, fully public domain
try:
    from datasets import load_dataset
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'datasets'], check=True)
    from datasets import load_dataset

console.print('[bold]Loading Enron email dataset...[/bold]')
ds = load_dataset('SetFit/enron_spam', split='train', trust_remote_code=True)
console.print(f'[green]✓[/green] Loaded {len(ds)} emails')

# ── Helper: extract email addresses from text ────────────────────────────────
def extract_emails(text):
    return list(set(re.findall(r'[\w.+-]+@[\w.-]+\.\w+', str(text or ''))))

def extract_name(email):
    """Best-effort name from email address."""
    local = email.split('@')[0]
    parts = re.split(r'[._-]', local)
    return ' '.join(p.capitalize() for p in parts if len(p) > 1)[:40]

def clean_body(text):
    """Strip forwarding headers and excessive whitespace."""
    text = re.sub(r'-{3,}.*?(?=\n\n|$)', '', str(text or ''), flags=re.DOTALL)
    text = re.sub(r'(From|To|Subject|Date|Cc):.*?\n', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:400]

# ── Ingest sample of emails ──────────────────────────────────────────────────
SAMPLE = 120   # number of emails to load (keep reasonable for embedding speed)
loaded, skipped = 0, 0
person_set = set()

for row in ds.select(range(min(SAMPLE * 3, len(ds)))):
    if loaded >= SAMPLE:
        break

    subject = str(row.get('subject') or row.get('Subject') or 'No Subject')[:120]
    body    = clean_body(row.get('message') or row.get('body') or row.get('text') or '')
    sender  = str(row.get('sender') or row.get('From') or '')[:80]
    label   = row.get('label', 0)  # 0=ham, 1=spam

    if not body or len(body) < 30:
        skipped += 1
        continue

    content = f"[{subject}] {body}"
    mtype   = 'conversation'
    tags    = ['enron', 'email', 'spam' if label else 'ham']
    if sender:
        tags.append(sender.split('@')[0][:20])

    # Extract entities mentioned
    ents = extract_emails(sender) + extract_emails(body)
    ents = [e for e in ents if 'enron' in e.lower() or len(e) < 40][:5]

    brain.store(content, memory_type=mtype, tags=tags,
                source='enron_dataset', entities_mentioned=ents[:3])
    loaded += 1

    # Track unique senders for entity graph
    if sender and '@' in sender:
        person_set.add(sender.strip())

console.print(f'[green]✓[/green] Loaded {loaded} emails ({skipped} skipped)')

# ── Build entity relationship graph from Enron org structure ─────────────────
console.print('[bold]Building Enron entity graph...[/bold]')

# Known Enron executives (public record)
enron_people = [
    ('Kenneth Lay',      'CEO'),
    ('Jeffrey Skilling', 'COO'),
    ('Andrew Fastow',    'CFO'),
    ('Sherron Watkins',  'VP Accounting'),
    ('Lou Pai',          'Chairman EES'),
    ('Rebecca Mark',     'VP International'),
]
enron_rels = [
    ('Kenneth Lay',      'Jeffrey Skilling', 'reports_to'),
    ('Jeffrey Skilling', 'Andrew Fastow',    'manages'),
    ('Andrew Fastow',    'Enron',            'works_at'),
    ('Kenneth Lay',      'Enron',            'leads'),
    ('Sherron Watkins',  'Kenneth Lay',      'reported_fraud_to'),
    ('Sherron Watkins',  'Enron',            'works_at'),
    ('Lou Pai',          'Enron',            'works_at'),
    ('Rebecca Mark',     'Enron',            'works_at'),
    ('Enron',            'Arthur Andersen',  'audited_by'),
    ('Enron',            'SEC',              'investigated_by'),
]

# Store key facts as memories
enron_facts = [
    ('Enron Corporation was an American energy company based in Houston, Texas.', 'fact', ['enron','company']),
    ('Enron filed for bankruptcy in December 2001 — one of the largest in US history.', 'event', ['enron','bankruptcy']),
    ('The Enron scandal involved systematic accounting fraud and off-balance-sheet entities.', 'fact', ['enron','fraud','accounting']),
    ('Kenneth Lay was Enron CEO and was convicted of fraud and conspiracy in 2006.', 'fact', ['enron','executives','fraud']),
    ('Jeffrey Skilling was COO/CEO of Enron and received a 24-year prison sentence.', 'fact', ['enron','executives','fraud']),
    ('Andrew Fastow was CFO and created the off-balance-sheet partnerships that hid debt.', 'fact', ['enron','executives','fraud']),
    ('Sherron Watkins was the Enron whistleblower who warned Kenneth Lay of accounting irregularities.', 'fact', ['enron','whistleblower']),
    ('Arthur Andersen, Enron\'s auditor, was convicted of obstruction of justice.', 'event', ['enron','andersen','auditor']),
    ('Enron\'s stock peaked at $90.75 in mid-2000 and collapsed to under $1 by late 2001.', 'fact', ['enron','stock','collapse']),
    ('The Enron email dataset contains ~500,000 emails from 150 employees made public by the FERC.', 'fact', ['enron','dataset','emails']),
]

for content, mtype, tags in enron_facts:
    brain.store(content, memory_type=mtype, tags=tags, source='enron_public_record')

for person, rel_type in enron_people:
    brain.store(f'{person} was {rel_type} at Enron Corporation.',
                memory_type='fact', tags=['enron', 'executive'],
                entities_mentioned=[person, 'Enron'])

for a, b, rel in enron_rels:
    brain.link_entities(a, b, rel)

console.print(f'[green]✓[/green] Enron knowledge graph built: {len(enron_people)} people, {len(enron_rels)} relations')
console.print(brain.stats())


Loading Enron email dataset...

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'SetFit/enron_spam' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Repo card metadata block was not found. Setting CardData to empty.


✓ Loaded 31716 emails

✓ Loaded 120 emails (0 skipped)

Building Enron entity graph...

...

↔ Rebecca Mark works_at Enron

↔ Enron audited_by Arthur Andersen

↔ Enron investigated_by SEC

✓ Enron knowledge graph built: 6 people, 10 relations

{'memories': 144, 'entities': 9, 'memory_edges': 299, 'entity_edges': 66, 'sessions': 2, 'daily_logs': 1}

## 7. `memory_search` — Semantic Recall


In [7]:
def pretty_search(query, top_k=5, memory_type=None):
    results = brain.search(query, top_k=top_k, memory_type=memory_type)
    table = Table(title=f'memory_search: "{query}"', show_lines=True)
    table.add_column('Score',   style='cyan',   width=8)
    table.add_column('Type',    style='yellow', width=14)
    table.add_column('Content', style='white')
    table.add_column('Tags',    style='dim',    width=22)
    for r in results:
        s = r.get('score') or 0
        table.add_row(f'{s:.3f}', r.get('memory_type',''), r.get('content','')[:110],
                      ', '.join(r.get('tags',[]))[:22])
    console.print(table)
    return results

pretty_search('accounting fraud')
pretty_search('bankruptcy 2001')
pretty_search('email communication', memory_type='conversation')


                                         memory_search: "accounting fraud"                                         
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Score    ┃ Type           ┃ Content                                                    ┃ Tags                   ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0.717    │ fact           │ The Enron scandal involved systematic accounting fraud and │ enron, fraud, accounti │
│          │                │ off-balance-sheet entities.                                │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.478    │ conversation   │ Sherron Watkins was the Enron whistleblower. She sent a    │                        │
│          │                │ memo to Kenneth Lay warning of accounting irregulariti     │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.470    │ conversation   │ Noted — review the Arthur Andersen audit trail for Enron.  │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.462    │ fact           │ Sherron Watkins was the Enron whistleblower who warned     │ enron, whistleblower   │
│          │                │ Kenneth Lay of accounting irregularities.                  │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.447    │ event          │ Arthur Andersen, Enron's auditor, was convicted of         │ enron, andersen, audit │
│          │                │ obstruction of justice.                                    │                        │
└──────────┴────────────────┴────────────────────────────────────────────────────────────┴────────────────────────┘

                                         memory_search: "bankruptcy 2001"                                          
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Score    ┃ Type           ┃ Content                                                    ┃ Tags                   ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0.647    │ event          │ Enron filed for Chapter 11 bankruptcy on December 2nd      │ enron, bankruptcy      │
│          │                │ 2001.                                                      │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.637    │ event          │ Enron filed for bankruptcy in December 2001 — one of the   │ enron, bankruptcy      │
│          │                │ largest in US history.                                     │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.402    │ conversation   │  if you have any questions i ' m sur                       │ enron, email, ham      │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.381    │ fact           │ Enron's stock peaked at $90.75 in mid-2000 and collapsed   │ enron, stock, collapse │
│          │                │ to under $1 by late 2001.                                  │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.360    │ conversation   │ Enron was an energy company that collapsed in 2001 due to  │                        │
│          │                │ systematic accounting fraud. Executives including Ke       │                        │
└──────────┴────────────────┴────────────────────────────────────────────────────────────┴────────────────────────┘

                                       memory_search: "email communication"                                        
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Score    ┃ Type           ┃ Content                                                    ┃ Tags                   ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0.472    │ conversation   │  the power of email marketing email marketing is           │ enron, email, spam     │
│          │                │ spreadingaround the wholeworld because                     │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.406    │ conversation   │  - these recipients of your message have been processed by │ enron, email, spam     │
│          │                │ the mail server : luc                                      │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.375    │ conversation   │  your message to : - > antonioa @ icb . ufmg . br was      │ enron, email, spam     │
│          │                │ considered u                                               │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.369    │ conversation   │  italiano : il suo messaggio non e '                       │ enron, email, spam     │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.354    │ conversation   │ [[ avfs ] romanian software production & export] to : avfs │ enron, email, spam     │
│          │                │ @ fazekas . hu attn : marketing department from : i        │                        │
└──────────┴────────────────┴────────────────────────────────────────────────────────────┴────────────────────────┘

[{'_id': 'memories/f395e61d25283064',
  '_key': 'f395e61d25283064',
  '_rev': '_lPoewlC--A',
  'access_count': 0,
  'agent_id': 'default',
  'confidence': 1,
  'content': "[promote your business] the power of email marketing email marketing is spreadingaround the wholeworld because of itshigh effectiveness , speedandlow cost . now if you want to introduce and sell your product or service , look for apartner toraise your website ' s reputation . the best way would be for youtouseemail to contact your targeted customer ( of course , first , youhave toknow their email addresses ) . targeted e",
  'created_at': '2026-03-23T12:17:18.436749+00:00',
  'embedding': [-0.021350620314478874,
   -0.05225292593240738,
   -0.011700017377734184,
   -0.025347616523504257,
   0.10653170198202133,
   -0.005622969940304756,
   0.060360733419656754,
   0.09365925192832947,
   0.02461426891386509,
   -0.051976241171360016,
   -0.006396493408828974,
   -0.02662450633943081,
   0.0612364336848259,
   0.01652

## 8. Session Replay — Store a Conversation


In [8]:
import uuid

session_id = str(uuid.uuid4())[:8]
brain.open_session(session_id, agent_id='default', channel='whatsapp')
console.print(f'[bold]Opened session:[/bold] {session_id}')

convo = [
    ('user',      'Tell me about the Enron scandal.'),
    ('assistant', 'Enron was an energy company that collapsed in 2001 due to systematic accounting fraud. '
                  'Executives including Kenneth Lay, Jeffrey Skilling, and Andrew Fastow were convicted.'),
    ('user',      'Who was the whistleblower?'),
    ('assistant', 'Sherron Watkins was the Enron whistleblower. She sent a memo to Kenneth Lay '
                  'warning of accounting irregularities in August 2001.'),
    ('user',      'Remember: I need to review the Arthur Andersen audit trail.'),
    ('assistant', 'Noted — review the Arthur Andersen audit trail for Enron.'),
]

for role, content in convo:
    brain.store_message(session_id, role=role, content=content)

brain.store('Review Arthur Andersen audit trail and Enron SPE documentation.',
            memory_type='decision', tags=['action','enron','audit'], session_id=session_id)

console.print(f'[green]✓[/green] Stored {len(convo)} messages + 1 durable fact')
pretty_search('Arthur Andersen audit', top_k=3)


Opened session: 506aa112

✓ Stored 6 messages + 1 durable fact

                                      memory_search: "Arthur Andersen audit"                                       
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Score    ┃ Type           ┃ Content                                                    ┃ Tags                   ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0.770    │ conversation   │ Noted — review the Arthur Andersen audit trail for Enron.  │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.762    │ conversation   │ Remember: I need to review the Arthur Andersen audit       │                        │
│          │                │ trail.                                                     │                        │
├──────────┼────────────────┼────────────────────────────────────────────────────────────┼────────────────────────┤
│ 0.688    │ decision       │ Review Arthur Andersen audit trail and Enron SPE           │ action, enron, audit   │
│          │                │ documentation.                                             │                        │
└──────────┴────────────────┴────────────────────────────────────────────────────────────┴────────────────────────┘

[{'_id': 'memories/0decf8f5722df008',
  '_key': '0decf8f5722df008',
  '_rev': '_lPoexnG--A',
  'access_count': 0,
  'agent_id': 'default',
  'confidence': 1,
  'content': 'Noted — review the Arthur Andersen audit trail for Enron.',
  'created_at': '2026-03-23T12:17:19.496724+00:00',
  'embedding': [-0.06587332487106323,
   0.10378871858119965,
   -0.07873380184173584,
   0.037023551762104034,
   0.043945327401161194,
   0.014054438099265099,
   0.04334314167499542,
   -0.018215829506516457,
   -0.019552698358893394,
   0.028909577056765556,
   0.026635702699422836,
   0.03588587045669556,
   0.010462760925292969,
   0.019436979666352272,
   -0.06811516731977463,
   -0.001263381796889007,
   0.011408769525587559,
   0.012406012043356895,
   0.07723025977611542,
   -0.038893017917871475,
   -0.005792593117803335,
   -0.057668380439281464,
   0.025562725961208344,
   -0.024437949061393738,
   0.03943189978599548,
   -0.005674224812537432,
   -0.008554225787520409,
   0.03367431089282036,


## 9. Entity Knowledge Graph — Graph Traversal


In [9]:
console.print('[bold]Entity neighbourhood: Daniel (depth=2)[/bold]')
neighbours = brain.entity_neighbourhood('Daniel', depth=2)

table = Table(title='Graph Neighbourhood', show_lines=True)
table.add_column('Depth',    width=6)
table.add_column('Relation', style='cyan', width=20)
table.add_column('Entity / Memory', style='white')
for n in neighbours[:20]:
    v = n['vertex']
    label = v.get('name') or v.get('content','')[:70]
    table.add_row(str(n['depth']), n.get('edge_label',''), label)
console.print(table)


Entity neighbourhood: Daniel (depth=2)

                Graph Neighbourhood                
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Depth  ┃ Relation             ┃ Entity / Memory ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
└────────┴──────────────────────┴─────────────────┘

## 10. Heartbeat Compaction — Daily Summary


In [10]:
daily = brain.compact_day(target_date=date.today())
if daily:
    console.print(f'\n[bold]Daily Summary ({daily["date"]}):[/bold]')
    print(daily['summary'])


✓ Compacted 143 memories → daily log: 2026-03-23

Daily Summary (2026-03-23):

# Daily Summary — 2026-03-23

## Conversation
- [any software just for 15 $ - 99 $] understanding oem software lead me not into temptation ; i can find the way myself . # 3533 . the law disregards trifles .
- [perspective on ferc regulatory action client conf call today , jun e] 19 th , 2 : 00 pm edt perspective on ferc regulatory action client conference call today , tuesday , june 19 th 2 : 00 pm edt host : ray niles , power / natural gas analyst speaker : steve bergstrom , president & coo of dynegy steve bergstrom , president and chief operating officer of dynegy , will join us at 2 : 00 p . m . today for a conference call discussion of the recent ferc action imposing 
- [wanted to try ci 4 lis but thought it was way too expensive for you ?] viagra at $ 1 . 12 per dose ready to boost your sex life ? positive ? time to do it right now . order viagra at incredibly low prices $ 1 . 12 per dose . unbelivable remove
- [enron / hpl actuals for december 11 , 2000] teco tap 30 . 000 / enron

## 11. OpenClaw Tool Shim

Drop-in `memory_*` functions matching OpenClaw's tool interface.
Wire these into a custom gateway tool or SKILL.md.


In [11]:
AGENT_ID = 'default'  # set to your openclaw agent id

def memory_store(content, memory_type='fact', tags=None):
    """Store a durable memory. Call when user says 'remember' or a key fact arises."""
    doc = brain.store(content, memory_type=memory_type, tags=tags or [], agent_id=AGENT_ID)
    return {'status': 'stored', 'key': doc['_key'], 'type': memory_type}

def memory_search(query, top_k=6, memory_type=None):
    """Semantic search over stored memories. Returns ranked snippets with source."""
    return [{
        'content': r['content'],
        'score':   round(r.get('score') or 0, 4),
        'type':    r['memory_type'],
        'source':  f'memories/{r["_key"]}',
        'tags':    r.get('tags', []),
        'created': r.get('created_at', ''),
    } for r in brain.search(query, top_k=top_k, memory_type=memory_type, agent_id=AGENT_ID)]

def memory_get(key):
    """Retrieve a memory by key. Returns {text, path} gracefully if missing."""
    doc = brain.get(key)
    if not doc: return {'text': '', 'path': key}
    return {'text': doc['content'], 'path': f'memories/{key}',
            'type': doc['memory_type'], 'tags': doc.get('tags', [])}

def memory_delete(key):
    ok = brain.delete(key)
    return {'status': 'deleted' if ok else 'not_found', 'key': key}


console.print('[bold]Testing tool shim...[/bold]')
r = memory_store('Enron filed for Chapter 11 bankruptcy on December 2nd 2001.',
                 memory_type='event', tags=['enron','bankruptcy'])
console.print(f'  memory_store  -> {r}')
hits = memory_search('Enron bankruptcy filing', top_k=3)
console.print(f'  memory_search -> {len(hits)} result(s)')
for h in hits:
    console.print(f'    [{h["score"]:.3f}] {h["content"][:80]}')
if hits:
    got = memory_get(hits[0]['source'].replace('memories/', ''))
    console.print(f'  memory_get    -> {got["text"][:70]}...')
console.print('[bold green]✓ Tool shim working[/bold green]')


Testing tool shim...

memory_store  -> {'status': 'stored', 'key': '30551c443c9aab0b', 'type': 'event'}

memory_search -> 3 result(s)

[0.820] Enron filed for bankruptcy in December 2001 — one of the largest in US history.

[0.810] Enron filed for Chapter 11 bankruptcy on December 2nd 2001.

[0.632]  enron seeks money for trading unit as customers flee ( updatel

memory_get    -> Enron filed for bankruptcy in December 2001 — one of the largest in US...

✓ Tool shim working

## 12. AQL Brain Inspector


In [12]:
def aql(query, bind_vars=None):
    return list(db.aql.execute(query, bind_vars=bind_vars or {}))

# Memory type breakdown
type_counts = aql('''
    FOR m IN memories
      COLLECT t = m.memory_type WITH COUNT INTO n
      SORT n DESC RETURN {type: t, count: n}
''')
table = Table(title='Memory Inventory')
table.add_column('Type', style='yellow')
table.add_column('Count', style='cyan', justify='right')
for row in type_counts: table.add_row(row['type'], str(row['count']))
console.print(table)

# Entity graph
rels = aql('''
    FOR e IN entity_edges
      LET f = DOCUMENT(e._from) LET t = DOCUMENT(e._to)
      RETURN {from: f.name, rel: e.relation, to: t.name || LEFT(t.content, 50)}
''')
table2 = Table(title='Entity Knowledge Graph', show_lines=True)
table2.add_column('From', style='green')
table2.add_column('Relation', style='cyan')
table2.add_column('To', style='blue')
for row in rels[:25]: table2.add_row(row.get('from','?'), row.get('rel',''), row.get('to','?'))
console.print(table2)

# Most accessed
top = aql('''
    FOR m IN memories SORT m.access_count DESC LIMIT 5
    RETURN {accesses: m.access_count, type: m.memory_type, content: LEFT(m.content,80)}
''')
table3 = Table(title='Most Accessed Memories', show_lines=True)
table3.add_column('Hits', style='cyan', width=6)
table3.add_column('Type', style='yellow', width=14)
table3.add_column('Content')
for row in top: table3.add_row(str(row['accesses']), row['type'], row['content'])
console.print(table3)


    Memory Inventory    
┏━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Type         ┃ Count ┃
┡━━━━━━━━━━━━━━╇━━━━━━━┩
│ conversation │   126 │
│ fact         │    14 │
│ event        │     3 │
│ decision     │     1 │
└──────────────┴───────┘

                                   Entity Knowledge Graph                                    
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ From             ┃ Relation          ┃ To                                                 ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Kenneth Lay      │ mentioned_in      │ Kenneth Lay was CEO at Enron Corporation.          │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Enron            │ mentioned_in      │ Kenneth Lay was CEO at Enron Corporation.          │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Jeffrey Skilling │ mentioned_in      │ Jeffrey Skilling was COO at Enron Corporation.     │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Enron            │ mentioned_in      │ Jeffrey Skilling was COO at Enron Corporation.     │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Andrew Fastow    │ mentioned_in      │ Andrew Fastow was CFO at Enron Corporation.        │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Enron            │ mentioned_in      │ Andrew Fastow was CFO at Enron Corporation.        │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Sherron Watkins  │ mentioned_in      │ Sherron Watkins was VP Accounting at Enron Corpora │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Enron            │ mentioned_in      │ Sherron Watkins was VP Accounting at Enron Corpora │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Lou Pai          │ mentioned_in      │ Lou Pai was Chairman EES at Enron Corporation.     │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Enron            │ mentioned_in      │ Lou Pai was Chairman EES at Enron Corporation.     │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Rebecca Mark     │ mentioned_in      │ Rebecca Mark was VP International at Enron Corpora │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Enron            │ mentioned_in      │ Rebecca Mark was VP International at Enron Corpora │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Kenneth Lay      │ reports_to        │ Jeffrey Skilling                                   │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Jeffrey Skilling │ manages           │ Andrew Fastow                                      │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Andrew Fastow    │ works_at          │ Enron                                              │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Kenneth Lay      │ leads             │ Enron                                              │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Sherron Watkins  │ reported_fraud_to │ Kenneth Lay                                        │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Sherron Watkins  │ works_at          │ Enron                                              │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Lou Pai          │ works_at          │ Enron                                              │
├──────────────────┼───────────────────┼────────────────────────────────────────────────────┤
│ Rebecca Mark     │ works_at          │ Enron      

                                            Most Accessed Memories                                            
┏━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Hits   ┃ Type           ┃ Content                                                                          ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 2      │ event          │ Enron filed for bankruptcy in December 2001 — one of the largest in US history.  │
├────────┼────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 1      │ fact           │ Sherron Watkins was the Enron whistleblower who warned Kenneth Lay of accounting │
├────────┼────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 1      │ event          │ Arthur Andersen, Enron's auditor, was convicted of obstruction of justice.       │
├────────┼────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 1      │ event          │ Enron filed for Chapter 11 bankruptcy on December 2nd 2001.                      │
├────────┼────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 1      │ conversation   │  enron seeks money for trading unit as customers flee ( updatel                  │
└────────┴────────────────┴──────────────────────────────────────────────────────────────────────────────────┘

## 13. SKILL.md Generator

Auto-generates the OpenClaw skill file for your agent.


In [13]:
skill_md = f'''
---
name: arango-brain
description: ArangoDB-backed digital brain - replaces SQLite memory with graph + vector + document storage
version: 1.0.0
---

# ArangoDB Digital Brain

Replaces OpenClaw's default SQLite memory layer with ArangoDB:
graph traversal, vector search, document storage, entity linking, and daily compaction.

## Connection
- Host:     {ARANGO_HOST}
- Database: {DB_NAME}

## Tools

### memory_store(content, memory_type, tags)
Store a durable memory. Call when:
- User says 'remember', 'note that', 'don't forget'
- A decision, preference, or key fact arises
- You want to persist something across sessions

memory_type: fact | event | note | conversation | decision | preference | daily_summary

### memory_search(query, top_k, memory_type)
Semantic search over all memories. Returns ranked list with source paths.
Always call this before saying 'I don't know' about something personal.

### memory_get(key)
Fetch a specific memory by key (from memory_search source field).

### memory_delete(key)
Remove a memory by key.

## Memory types guide
| Type | Use for |
|------|--------|
| fact | Timeless facts about user, world, projects |
| event | Things that happened at a point in time |
| decision | Architectural, personal or strategic choices |
| preference | User preferences and settings |
| note | General reminders and to-dos |
| conversation | Session message turns |
| daily_summary | Auto-generated daily compaction entries |

## Source citation
memory_search results include source: memories/<key>.
You can reference this as Source: memories/<key> in responses.
'''

with open('/tmp/SKILL_arango_brain.md', 'w') as f:
    f.write(skill_md)
console.print('[green]✓[/green] SKILL.md written to /tmp/SKILL_arango_brain.md')
print(skill_md)


✓ SKILL.md written to /tmp/SKILL_arango_brain.md


---
name: arango-brain
description: ArangoDB-backed digital brain - replaces SQLite memory with graph + vector + document storage
version: 1.0.0
---

# ArangoDB Digital Brain

Replaces OpenClaw's default SQLite memory layer with ArangoDB:
graph traversal, vector search, document storage, entity linking, and daily compaction.

## Connection
- Host:     https://5ieeavs2.rnd.pilot.arango.ai
- Database: openclaw_brain

## Tools

### memory_store(content, memory_type, tags)
Store a durable memory. Call when:
- User says 'remember', 'note that', 'don't forget'
- A decision, preference, or key fact arises
- You want to persist something across sessions

memory_type: fact | event | note | conversation | decision | preference | daily_summary

### memory_search(query, top_k, memory_type)
Semantic search over all memories. Returns ranked list with source paths.
Always call this before saying 'I don't know' about something personal.

### memory_get(key)
Fetch a specific memory by key (from memory

## 14. Health Check


In [14]:
console.print('[bold]─── OpenClaw × ArangoDB Brain — Health Check ───[/bold]')
stats = brain.stats()
checks = [
    ('ArangoDB connection',         True),
    ('Schema (collections+graph)',  True),
    ('Embedding model loaded',      EMBED_MODEL is not None),
    ('Native vector index',         VECTOR_INDEX_NATIVE),
    ('Memories stored',             stats['memories'] > 0),
    ('Entities stored',             stats['entities'] > 0),
    ('Entity edges',                stats['entity_edges'] > 0),
    ('Sessions tracked',            stats['sessions'] > 0),
]
for label, ok in checks:
    icon = '[green]✓[/green]' if ok else '[yellow]○[/yellow]'
    console.print(f'  {icon} {label}')
console.print('\n[bold]Collection counts:[/bold]')
for k, v in stats.items(): console.print(f'  {k:<16} {v}')
console.print('\n[bold green]🦞 OpenClaw Digital Brain is live on ArangoDB[/bold green]')


─── OpenClaw × ArangoDB Brain — Health Check ───

✓ ArangoDB connection

✓ Schema (collections+graph)

✓ Embedding model loaded

✓ Native vector index

...

memory_edges     448

entity_edges     66

sessions         3

daily_logs       2

🦞 OpenClaw Digital Brain is live on ArangoDB

In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# 🦞  OPENCLAW × ARANGODB — DIGITAL BRAIN UI  (v8 - clean rewrite)
# ═══════════════════════════════════════════════════════════════════════════

from IPython.display import display, HTML as _IHTML
import json as _json, base64 as _b64

_mem = list(db.aql.execute(
    "FOR m IN memories SORT m.created_at DESC LIMIT 200 "
    "RETURN {_key:m._key,content:m.content,type:m.memory_type,"
    "tags:m.tags,created:m.created_at,access:m.access_count}"
))
_ent = list(db.aql.execute(
    "FOR e IN entity_edges "
    "LET f=DOCUMENT(e._from) LET t=DOCUMENT(e._to) "
    "RETURN {fn:f.name,tn:t.name||LEFT(t.content,40),rel:e.relation}"
))
_st = brain.stats()
_bt = list(db.aql.execute(
    "FOR m IN memories COLLECT t=m.memory_type WITH COUNT INTO n "
    "SORT n DESC RETURN {type:t,count:n}"
))
_tr = list(db.aql.execute(
    "FOR m IN memories LET d=LEFT(m.created_at,10) "
    "COLLECT day=d WITH COUNT INTO n SORT day DESC LIMIT 7 RETURN {day:day,count:n}"
))
_DATA = _json.dumps({
    "memories":_mem,"entities":_ent,
    "stats":{"totals":_st,"by_type":_bt,"trend":_tr}
}, ensure_ascii=True)
_T = '<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="UTF-8"/>\n<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=JetBrains+Mono:wght@300;400;500&display=swap" rel="stylesheet"/>\n<script src="https://cdnjs.cloudflare.com/ajax/libs/d3/7.8.5/d3.min.js"></script>\n<style>\n:root{--bg:#090d12;--bg1:#0e1318;--bg2:#141b24;--bg3:#1a2535;--bo:#1d2b3a;--bo2:#28394f;--ac:#00c17c;--a2:#0088ee;--a3:#e84d2a;--a4:#f5a623;--tx:#dce8f2;--tx2:#7a8fa8;--tx3:#3d5068;--mo:\'JetBrains Mono\',monospace;--sa:\'Syne\',sans-serif}\n*{box-sizing:border-box;margin:0;padding:0}\nbody{background:var(--bg);color:var(--tx);font-family:var(--sa);min-height:100vh}\n::-webkit-scrollbar{width:4px}::-webkit-scrollbar-track{background:transparent}::-webkit-scrollbar-thumb{background:var(--bo2);border-radius:4px}\n\n.hdr{display:flex;align-items:center;justify-content:space-between;padding:14px 24px;background:var(--bg1);border-bottom:1px solid var(--bo);position:sticky;top:0;z-index:50}\n.brand{display:flex;align-items:center;gap:10px}\n.brand-ico{font-size:20px;animation:glow 3s ease-in-out infinite}\n@keyframes glow{0%,100%{filter:drop-shadow(0 0 4px #00c17c50)}50%{filter:drop-shadow(0 0 12px #00c17c)}}\n.brand-name{font-size:14px;font-weight:800;letter-spacing:.04em}\n.brand-name b{color:var(--ac)}\n.brand-sub{font-size:9px;color:var(--tx3);font-family:var(--mo);letter-spacing:.12em}\n.hstats{display:flex;gap:20px}\n.hs{text-align:right}\n.hs-v{font-family:var(--mo);font-size:18px;color:var(--ac);line-height:1}\n.hs-l{font-size:9px;color:var(--tx3);text-transform:uppercase;letter-spacing:.1em;margin-top:2px}\n\n.nav{display:flex;padding:0 24px;background:var(--bg1);border-bottom:1px solid var(--bo)}\n.tab{padding:11px 16px;font-size:11px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;cursor:pointer;border:none;background:none;color:var(--tx3);border-bottom:2px solid transparent;transition:all .2s}\n.tab:hover{color:var(--tx2)}\n.tab.on{color:var(--ac);border-bottom-color:var(--ac)}\n\n.main{padding:20px 24px}\n.panel{display:none}.panel.on{display:block}\n\n.card{background:var(--bg1);border:1px solid var(--bo);border-radius:6px;padding:16px;margin-bottom:14px}\n.ctitle{font-size:10px;font-weight:700;letter-spacing:.1em;text-transform:uppercase;color:var(--tx3);margin-bottom:14px;display:flex;align-items:center;gap:6px}\n.ctitle::before{content:\'\';display:inline-block;width:2px;height:10px;background:var(--ac);border-radius:2px}\n\n.sbar{display:flex;gap:8px;background:var(--bg2);border:1px solid var(--bo2);border-radius:6px;padding:3px 3px 3px 14px;margin-bottom:12px;transition:border-color .2s;align-items:center}\n.sbar:focus-within{border-color:var(--ac)}\n.sinp{flex:1;background:none;border:none;outline:none;color:var(--tx);font-family:var(--mo);font-size:13px;padding:9px 0}\n.sinp::placeholder{color:var(--tx3)}\n.btn{padding:9px 16px;border:none;border-radius:5px;cursor:pointer;font-family:var(--sa);font-size:11px;font-weight:700;letter-spacing:.06em;text-transform:uppercase;transition:all .15s}\n.btn-g{background:var(--ac);color:#000}.btn-g:hover{background:#00e090}\n.btn-r{background:transparent;border:1px solid var(--a3);color:var(--a3)}.btn-r:hover{background:var(--a3);color:#fff}\n\n.chips{display:flex;gap:6px;flex-wrap:wrap;align-items:center;margin-bottom:12px}\n.chip{padding:4px 11px;border-radius:20px;font-size:10px;font-weight:700;cursor:pointer;border:1px solid var(--bo2);background:var(--bg2);color:var(--tx3);transition:all .15s}\n.chip:hover,.chip.on{background:var(--ac);color:#000;border-color:var(--ac)}\n\n.rgrid{display:grid;grid-template-columns:repeat(auto-fill,minmax(300px,1fr));gap:10px;margin-top:4px}\n.mc{background:var(--bg2);border:1px solid var(--bo);border-radius:6px;padding:14px;cursor:pointer;transition:all .2s;position:relative;overflow:hidden}\n.mc::before{content:\'\';position:absolute;left:0;top:0;bottom:0;width:3px;background:var(--ac);opacity:0;transition:opacity .2s}\n.mc:hover{border-color:var(--bo2);transform:translateY(-1px);box-shadow:0 4px 16px #00000040}\n.mc:hover::before,.mc.sel::before{opacity:1}\n.mc.sel{border-color:var(--ac);background:var(--bg3)}\n.mt{display:inline-block;padding:2px 8px;border-radius:3px;font-size:9px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;margin-bottom:8px}\n.mc-txt{font-family:var(--mo);font-size:11px;color:var(--tx);line-height:1.65;margin-bottom:8px}\n.mc-meta{display:flex;gap:7px;align-items:center;flex-wrap:wrap}\n.mc-score{font-family:var(--mo);font-size:10px;padding:2px 6px;background:var(--bg3);border-radius:3px;color:var(--ac)}\n.mc-tag{font-size:9px;padding:2px 6px;border-radius:3px;background:var(--bg3);color:var(--tx3);border:1px solid var(--bo)}\n.mc-date{font-size:9px;color:var(--tx3);font-family:var(--mo);margin-left:auto}\n.mc-key{font-size:8px;color:var(--tx3);font-family:var(--mo);margin-top:4px;opacity:.4}\n.nores{text-align:center;padding:50px 20px;color:var(--tx3);font-family:var(--mo);font-size:12px;grid-column:1/-1}\n\n.t-fact{background:#00c17c18;color:#00c17c}\n.t-event{background:#0088ee18;color:#5599ff}\n.t-decision{background:#e84d2a18;color:#e84d2a}\n.t-preference{background:#f5a62318;color:#f5a623}\n.t-note{background:#7a8fa818;color:#7a8fa8}\n.t-conversation{background:#aa44ff18;color:#aa44ff}\n.t-daily_summary{background:#00c17c10;color:#00c17c}\n\n.dpane{position:fixed;right:0;top:0;bottom:0;width:380px;background:var(--bg1);border-left:1px solid var(--bo);padding:20px;overflow-y:auto;transform:translateX(100%);transition:transform .25s cubic-bezier(.4,0,.2,1);z-index:200}\n.dpane.open{transform:translateX(0)}\n.dclose{position:absolute;top:12px;right:12px;background:var(--bg2);border:1px solid var(--bo);color:var(--tx2);cursor:pointer;font-size:13px;padding:5px 9px;border-radius:4px}\n.dfield{margin-bottom:12px}\n.dfl{font-size:9px;color:var(--tx3);text-transform:uppercase;letter-spacing:.1em;margin-bottom:4px}\n.dfv{font-family:var(--mo);font-size:11px;color:var(--tx);background:var(--bg2);padding:9px 11px;border-radius:4px;line-height:1.6;border:1px solid var(--bo);word-break:break-word}\n\n.sgrid2{display:grid;grid-template-columns:1fr 270px;gap:14px;align-items:start}\n.inp,.sel,.ta{width:100%;background:var(--bg2);border:1px solid var(--bo2);border-radius:6px;color:var(--tx);font-family:var(--mo);font-size:13px;padding:10px 12px;outline:none;transition:border-color .2s}\n.inp:focus,.sel:focus,.ta:focus{border-color:var(--ac)}\n.ta{min-height:100px;resize:vertical;line-height:1.6}\n.sel option{background:var(--bg2)}\n.lbl{display:block;font-size:10px;color:var(--tx3);text-transform:uppercase;letter-spacing:.08em;margin-bottom:6px}\n.fg{margin-bottom:12px}\n.rlist{display:flex;flex-direction:column;gap:7px}\n.ri{padding:9px 11px;background:var(--bg2);border-radius:4px;border:1px solid var(--bo)}\n.ric{font-size:11px;font-family:var(--mo);color:var(--tx);line-height:1.5;margin-bottom:3px;overflow:hidden;display:-webkit-box;-webkit-line-clamp:2;-webkit-box-orient:vertical}\n.rim{font-size:9px;color:var(--tx3)}\n\n.kpis{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-bottom:16px}\n.kpi{background:var(--bg1);border:1px solid var(--bo);border-radius:6px;padding:16px;text-align:center;position:relative;overflow:hidden}\n.kpi::after{content:\'\';position:absolute;bottom:0;left:0;right:0;height:2px;background:linear-gradient(90deg,var(--ac),var(--a2))}\n.knum{font-family:var(--mo);font-size:26px;font-weight:500;color:var(--ac);line-height:1;margin-bottom:4px}\n.klbl{font-size:9px;color:var(--tx3);text-transform:uppercase;letter-spacing:.1em}\n.charts{display:grid;grid-template-columns:1fr 1fr;gap:14px}\n.bi{display:flex;align-items:center;gap:8px;margin-bottom:7px}\n.bl{font-size:10px;color:var(--tx2);width:110px;font-family:var(--mo);text-align:right;flex-shrink:0}\n.btr{flex:1;background:var(--bg3);border-radius:2px;height:7px;overflow:hidden}\n.bf{height:100%;border-radius:2px;background:linear-gradient(90deg,var(--ac),var(--a2));transition:width 1s cubic-bezier(.4,0,.2,1)}\n.bc{font-size:10px;color:var(--tx3);font-family:var(--mo);width:24px;text-align:right}\n.tbars{display:flex;align-items:flex-end;gap:5px;height:100px}\n.tbar{flex:1;border-radius:3px 3px 0 0;background:linear-gradient(180deg,var(--ac),var(--a2));min-height:3px}\n.tbar:hover{filter:brightness(1.3)}\n.tlbls{display:flex;gap:5px;margin-top:5px}\n.tlbl{flex:1;text-align:center;font-size:8px;color:var(--tx3);font-family:var(--mo)}\n\n.gwrap{background:var(--bg1);border:1px solid var(--bo);border-radius:6px;overflow:hidden;position:relative}\n.gctrl{position:absolute;top:10px;left:10px;z-index:10;display:flex;flex-direction:column;gap:5px}\n.gbtn{width:30px;height:30px;display:flex;align-items:center;justify-content:center;background:var(--bg2);border:1px solid var(--bo2);border-radius:4px;cursor:pointer;font-size:14px;color:var(--tx2);user-select:none}\n.gbtn:hover{background:var(--bg3);color:var(--tx)}\n.gleg{position:absolute;bottom:10px;left:10px;background:var(--bg2);border:1px solid var(--bo);border-radius:4px;padding:8px 10px}\n.gli{display:flex;align-items:center;gap:6px;font-size:10px;color:var(--tx2);margin-bottom:2px}\n.gli:last-child{margin:0}\n.gld{width:8px;height:8px;border-radius:50%}\n.gtip{position:fixed;padding:6px 10px;background:var(--bg3);border:1px solid var(--bo2);border-radius:4px;font-size:11px;font-family:var(--mo);color:var(--tx);pointer-events:none;z-index:300;display:none;max-width:180px}\n\n.toast{position:fixed;bottom:20px;right:20px;padding:10px 16px;background:var(--ac);color:#000;border-radius:6px;font-weight:700;font-size:12px;z-index:999;animation:sup .2s ease,fout .3s ease 1.7s forwards;pointer-events:none}\n.toast.err{background:var(--a3);color:#fff}\n@keyframes sup{from{transform:translateY(12px);opacity:0}to{transform:translateY(0);opacity:1}}\n@keyframes fout{to{opacity:0}}\n@media(max-width:800px){.sgrid2{grid-template-columns:1fr}.charts{grid-template-columns:1fr}.kpis{grid-template-columns:1fr 1fr}.rgrid{grid-template-columns:1fr}}\n</style>\n</head>\n<body>\n\n<div class="hdr">\n  <div class="brand">\n    <span class="brand-ico">&#x1F99E;</span>\n    <div>\n      <div class="brand-name">OpenClaw <b>x</b> ArangoDB</div>\n      <div class="brand-sub">DIGITAL BRAIN</div>\n    </div>\n  </div>\n  <div class="hstats">\n    <div class="hs"><div class="hs-v" id="hm">0</div><div class="hs-l">Memories</div></div>\n    <div class="hs"><div class="hs-v" id="he">0</div><div class="hs-l">Entities</div></div>\n    <div class="hs"><div class="hs-v" id="hg">0</div><div class="hs-l">Edges</div></div>\n    <div class="hs"><div class="hs-v" id="hs2">0</div><div class="hs-l">Sessions</div></div>\n  </div>\n</div>\n\n<div class="nav">\n  <button class="tab on" onclick="go(\'search\',this)">&#128269; Search</button>\n  <button class="tab" onclick="go(\'store\',this)">&#9999; Store</button>\n  <button class="tab" onclick="go(\'graph\',this)">&#128375; Graph</button>\n  <button class="tab" onclick="go(\'stats\',this)">&#128202; Stats</button>\n</div>\n\n<div class="main">\n\n  <div id="p-search" class="panel on">\n    <div class="sbar">\n      <input class="sinp" id="sinp" placeholder="Search memories..." onkeydown="if(event.key===\'Enter\')doSearch()"/>\n      <button class="btn btn-g" onclick="doSearch()">Search</button>\n    </div>\n    <div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:8px;flex-wrap:wrap;gap:8px">\n      <div class="chips" style="margin:0" id="tchips">\n        <span class="chip on" onclick="ftype(this,\'\')">All</span>\n        <span class="chip" onclick="ftype(this,\'fact\')">Fact</span>\n        <span class="chip" onclick="ftype(this,\'event\')">Event</span>\n        <span class="chip" onclick="ftype(this,\'decision\')">Decision</span>\n        <span class="chip" onclick="ftype(this,\'preference\')">Preference</span>\n        <span class="chip" onclick="ftype(this,\'note\')">Note</span>\n        <span class="chip" onclick="ftype(this,\'conversation\')">Conv</span>\n      </div>\n      <select class="sel" id="topk" style="width:80px;padding:6px 8px;font-size:11px" onchange="doSearch()">\n        <option>5</option><option selected>8</option><option>15</option><option>30</option>\n      </select>\n    </div>\n    <div id="sres" class="rgrid"></div>\n  </div>\n\n  <div id="p-store" class="panel">\n    <div class="sgrid2">\n      <div class="card">\n        <div class="ctitle">New Memory</div>\n        <div class="fg"><label class="lbl">Content</label>\n          <textarea class="ta" id="sc" placeholder="What should the brain remember?"></textarea></div>\n        <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px">\n          <div class="fg"><label class="lbl">Type</label>\n            <select class="sel" id="stype">\n              <option value="fact">Fact</option><option value="event">Event</option>\n              <option value="decision">Decision</option><option value="preference">Preference</option>\n              <option value="note">Note</option><option value="conversation">Conversation</option>\n            </select></div>\n          <div class="fg"><label class="lbl">Tags (comma-sep)</label>\n            <input class="inp" id="stags" placeholder="tag1, tag2..."/></div>\n        </div>\n        <button class="btn btn-g" style="width:100%;padding:11px" onclick="doStore()">Store Memory</button>\n      </div>\n      <div class="card">\n        <div class="ctitle">Recent Memories</div>\n        <div class="rlist" id="rlist"></div>\n      </div>\n    </div>\n  </div>\n\n  <div id="p-graph" class="panel">\n    <div class="ctitle" style="margin-bottom:10px">Entity Knowledge Graph</div>\n    <div class="gwrap" id="gwrap">\n      <div class="gctrl">\n        <div class="gbtn" onclick="gzi()">+</div>\n        <div class="gbtn" onclick="gzo()">-</div>\n        <div class="gbtn" onclick="gzr()">o</div>\n      </div>\n      <svg id="gsvg"></svg>\n      <div class="gleg">\n        <div class="gli"><div class="gld" style="background:#00c17c"></div>Entity</div>\n        <div class="gli"><div class="gld" style="background:#5599ff"></div>Memory ref</div>\n      </div>\n    </div>\n    <div class="gtip" id="gtip"></div>\n  </div>\n\n  <div id="p-stats" class="panel">\n    <div class="kpis" id="kpis"></div>\n    <div class="charts">\n      <div class="card"><div class="ctitle">By Type</div><div id="barchart"></div></div>\n      <div class="card"><div class="ctitle">Last 7 Days</div>\n        <div class="tbars" id="trbars"></div>\n        <div class="tlbls" id="trlbls"></div>\n      </div>\n    </div>\n  </div>\n\n</div>\n\n<div class="dpane" id="dpane">\n  <button class="dclose" onclick="document.getElementById(\'dpane\').classList.remove(\'open\')">x</button>\n  <div id="dcont"></div>\n</div>\n\n<script type="application/json" id="__braindata__">__JSONDATA__</script>\n<script>\nvar D = JSON.parse(document.getElementById(\'__braindata__\').textContent);\nvar curType = \'\', selKey = null, zb = null;\n\n// ── header ────────────────────────────────────────────────────────────────────\nvar t = D.stats.totals;\ndocument.getElementById(\'hm\').textContent  = t.memories  || 0;\ndocument.getElementById(\'he\').textContent  = t.entities  || 0;\ndocument.getElementById(\'hg\').textContent  = t.entity_edges || 0;\ndocument.getElementById(\'hs2\').textContent = t.sessions  || 0;\n\n// ── render memories on load ───────────────────────────────────────────────────\nshowMems(D.memories.slice(0, 8));\nloadRecent();\n\n// ── tabs ──────────────────────────────────────────────────────────────────────\nfunction go(name, btn) {\n  document.querySelectorAll(\'.tab\').forEach(function(t){ t.classList.remove(\'on\'); });\n  btn.classList.add(\'on\');\n  document.querySelectorAll(\'.panel\').forEach(function(p){ p.classList.remove(\'on\'); });\n  document.getElementById(\'p-\' + name).classList.add(\'on\');\n  if (name === \'graph\') setTimeout(initGraph, 60);\n  if (name === \'stats\') renderStats();\n}\n\n// ── search ────────────────────────────────────────────────────────────────────\nfunction doSearch() {\n  var q   = document.getElementById(\'sinp\').value.trim();\n  var k   = parseInt(document.getElementById(\'topk\').value, 10);\n  var pool = curType ? D.memories.filter(function(m){ return m.type === curType; }) : D.memories;\n  if (!q) { showMems(pool.slice(0, k)); return; }\n  var ql   = q.toLowerCase();\n  var words = ql.split(/\\s+/).filter(function(w){ return w.length > 2; });\n  var scored = pool.map(function(m) {\n    var c = (m.content || \'\').toLowerCase();\n    var s = 0;\n    if (c.indexOf(ql) > -1) s = 0.97;\n    else {\n      for (var i = 0; i < words.length; i++) { if (c.indexOf(words[i]) > -1) { s = Math.max(s, 0.7); } }\n      var tags = m.tags || [];\n      for (var j = 0; j < tags.length; j++) { if ((tags[j]||\'\').toLowerCase().indexOf(ql) > -1) s = Math.max(s, 0.6); }\n    }\n    return { _key: m._key, content: m.content, type: m.type, tags: m.tags, created: m.created, access: m.access, score: s };\n  }).filter(function(m){ return m.score > 0; })\n    .sort(function(a, b){ return b.score - a.score; })\n    .slice(0, k);\n  showMems(scored.length ? scored : pool.slice(0, k));\n}\n\nfunction ftype(el, t) {\n  curType = t;\n  document.querySelectorAll(\'.chip\').forEach(function(c){ c.classList.remove(\'on\'); });\n  el.classList.add(\'on\');\n  doSearch();\n}\n\nfunction showMems(items) {\n  var el = document.getElementById(\'sres\');\n  if (!items || !items.length) { el.innerHTML = \'<div class="nores">No memories found.</div>\'; return; }\n  var html = \'\';\n  for (var i = 0; i < items.length; i++) {\n    var m = items[i];\n    var key  = m._key || \'\';\n    var type = m.type || \'note\';\n    var txt  = (m.content || \'\').slice(0, 130);\n    if ((m.content||\'\').length > 130) txt += \'...\';\n    var score = (m.score > 0) ? \'<span class="mc-score">\' + m.score.toFixed(3) + \'</span>\' : \'\';\n    var tags  = \'\';\n    var ta = m.tags || [];\n    for (var j = 0; j < Math.min(ta.length, 3); j++) {\n      tags += \'<span class="mc-tag">\' + esc(ta[j]) + \'</span>\';\n    }\n    var date = fd(m.created || \'\');\n    html += \'<div class="mc\' + (key === selKey ? \' sel\' : \'\') + \'" onclick="selMem(\\\'\' + key + \'\\\',this)">\'\n          + \'<div class="mt t-\' + type + \'">\' + type + \'</div>\'\n          + \'<div class="mc-txt">\' + esc(txt) + \'</div>\'\n          + \'<div class="mc-meta">\' + score + tags + \'<span class="mc-date">\' + date + \'</span></div>\'\n          + \'<div class="mc-key">\' + key + \'</div>\'\n          + \'</div>\';\n  }\n  el.innerHTML = html;\n}\n\nfunction selMem(key, el) {\n  document.querySelectorAll(\'.mc\').forEach(function(c){ c.classList.remove(\'sel\'); });\n  el.classList.add(\'sel\');\n  selKey = key;\n  var m = null;\n  for (var i = 0; i < D.memories.length; i++) { if (D.memories[i]._key === key) { m = D.memories[i]; break; } }\n  if (!m) return;\n  var score = m.score > 0 ? \'<div class="dfield"><div class="dfl">Score</div><div class="dfv">\' + m.score.toFixed(4) + \'</div></div>\' : \'\';\n  document.getElementById(\'dcont\').innerHTML =\n    \'<div style="margin-bottom:14px"><span class="mt t-\' + (m.type||\'note\') + \'">\' + (m.type||\'note\') + \'</span></div>\'\n    + \'<div class="dfield"><div class="dfl">Content</div><div class="dfv">\' + esc(m.content||\'\') + \'</div></div>\'\n    + score\n    + \'<div class="dfield"><div class="dfl">Tags</div><div class="dfv">\' + ((m.tags||[]).join(\', \')||\'none\') + \'</div></div>\'\n    + \'<div class="dfield"><div class="dfl">Key</div><div class="dfv">\' + key + \'</div></div>\'\n    + \'<div class="dfield"><div class="dfl">Created</div><div class="dfv">\' + fdf(m.created||\'\') + \'</div></div>\'\n    + \'<div class="dfield"><div class="dfl">Accesses</div><div class="dfv">\' + (m.access||0) + \'</div></div>\'\n    + \'<div style="margin-top:14px"><button class="btn btn-r" style="width:100%" onclick="doDel(\\\'\' + key + \'\\\')">Delete</button></div>\';\n  document.getElementById(\'dpane\').classList.add(\'open\');\n}\n\nfunction doDel(key) {\n  if (!key || !confirm(\'Delete?\')) return;\n  for (var i = 0; i < D.memories.length; i++) { if (D.memories[i]._key === key) { D.memories.splice(i,1); break; } }\n  document.getElementById(\'dpane\').classList.remove(\'open\');\n  doSearch();\n  document.getElementById(\'hm\').textContent = D.memories.length;\n  toast(\'Deleted\');\n}\n\n// ── store ─────────────────────────────────────────────────────────────────────\nfunction loadRecent() {\n  var html = \'\';\n  var items = D.memories.slice(0, 6);\n  for (var i = 0; i < items.length; i++) {\n    var m = items[i];\n    html += \'<div class="ri"><div class="ric">\' + esc(m.content||\'\') + \'</div>\'\n          + \'<div class="rim"><span class="mt t-\' + (m.type||\'note\') + \'" style="font-size:8px;padding:1px 5px">\' + (m.type||\'note\') + \'</span> \' + fd(m.created||\'\') + \'</div></div>\';\n  }\n  document.getElementById(\'rlist\').innerHTML = html || \'<div style="color:var(--tx3);font-size:11px;font-family:var(--mo)">No memories yet</div>\';\n}\n\nfunction doStore() {\n  var c = document.getElementById(\'sc\').value.trim();\n  if (!c) { toast(\'Content required\', true); return; }\n  var ty = document.getElementById(\'stype\').value;\n  var rawTags = document.getElementById(\'stags\').value.split(\',\');\n  var tags = [];\n  for (var i = 0; i < rawTags.length; i++) { var t2 = rawTags[i].trim(); if (t2) tags.push(t2); }\n  var m = { _key: Math.random().toString(36).slice(2,10), content: c, type: ty, tags: tags,\n            created: new Date().toISOString(), access: 0, score: 0 };\n  D.memories.unshift(m);\n  document.getElementById(\'sc\').value = \'\';\n  document.getElementById(\'stags\').value = \'\';\n  loadRecent();\n  document.getElementById(\'hm\').textContent = D.memories.length;\n  toast(\'Stored!\');\n}\n\n// ── graph ─────────────────────────────────────────────────────────────────────\nfunction initGraph() {\n  var wrap = document.getElementById(\'gwrap\');\n  var W = wrap.clientWidth || 800;\n  var H = Math.max(500, window.innerHeight * 0.6);\n  var svgEl = document.getElementById(\'gsvg\');\n  svgEl.setAttribute(\'width\', W); svgEl.setAttribute(\'height\', H);\n  d3.select(\'#gsvg\').selectAll(\'*\').remove();\n  var svg = d3.select(\'#gsvg\'), g = svg.append(\'g\');\n  zb = d3.zoom().scaleExtent([0.1, 5]).on(\'zoom\', function(e){ g.attr(\'transform\', e.transform); });\n  svg.call(zb);\n  var nm = {};\n  var links = [];\n  for (var i = 0; i < D.entities.length; i++) {\n    var e = D.entities[i];\n    if (e.fn && !nm[e.fn]) nm[e.fn] = { id: e.fn, t: \'entity\' };\n    if (e.tn && !nm[e.tn]) nm[e.tn] = { id: e.tn, t: (e.tn.length > 30 ? \'memory\' : \'entity\') };\n    if (e.fn && e.tn) links.push({ source: e.fn, target: e.tn, rel: e.rel||\'\' });\n  }\n  var nodes = Object.values(nm);\n  if (!nodes.length) {\n    g.append(\'text\').attr(\'x\', W/2).attr(\'y\', H/2).attr(\'text-anchor\',\'middle\')\n     .attr(\'fill\',\'#3d5068\').attr(\'font-size\',13).attr(\'font-family\',\'JetBrains Mono,monospace\')\n     .text(\'No entity edges to display\');\n    return;\n  }\n  svg.append(\'defs\').append(\'marker\').attr(\'id\',\'arr\').attr(\'viewBox\',\'0 -3 7 7\')\n    .attr(\'refX\',13).attr(\'markerWidth\',5).attr(\'markerHeight\',5).attr(\'orient\',\'auto\')\n    .append(\'path\').attr(\'d\',\'M0,-3L7,0L0,3\').attr(\'fill\',\'#28394f\');\n  var sim = d3.forceSimulation(nodes)\n    .force(\'link\', d3.forceLink(links).id(function(d){ return d.id; }).distance(90))\n    .force(\'charge\', d3.forceManyBody().strength(-200))\n    .force(\'center\', d3.forceCenter(W/2, H/2))\n    .force(\'col\', d3.forceCollide(22));\n  var link = g.append(\'g\').selectAll(\'line\').data(links).join(\'line\')\n    .attr(\'stroke\',\'#28394f\').attr(\'stroke-width\',1.5).attr(\'marker-end\',\'url(#arr)\');\n  var llbl = g.append(\'g\').selectAll(\'text\').data(links).join(\'text\')\n    .text(function(d){ return d.rel; }).attr(\'fill\',\'#3d5068\').attr(\'font-size\',8)\n    .attr(\'font-family\',\'JetBrains Mono,monospace\').attr(\'text-anchor\',\'middle\');\n  var tip = document.getElementById(\'gtip\');\n  var node = g.append(\'g\').selectAll(\'g\').data(nodes).join(\'g\')\n    .call(d3.drag()\n      .on(\'start\', function(e,d){ if(!e.active) sim.alphaTarget(0.3).restart(); d.fx=d.x; d.fy=d.y; })\n      .on(\'drag\',  function(e,d){ d.fx=e.x; d.fy=e.y; })\n      .on(\'end\',   function(e,d){ if(!e.active) sim.alphaTarget(0); d.fx=null; d.fy=null; }))\n    .on(\'mouseover\', function(e,d){ tip.style.display=\'block\'; tip.style.left=(e.clientX+10)+\'px\'; tip.style.top=(e.clientY-6)+\'px\'; tip.textContent=d.id; })\n    .on(\'mousemove\', function(e){ tip.style.left=(e.clientX+10)+\'px\'; tip.style.top=(e.clientY-6)+\'px\'; })\n    .on(\'mouseout\',  function(){ tip.style.display=\'none\'; });\n  node.append(\'circle\')\n    .attr(\'r\', function(d){ return d.t===\'entity\' ? 12 : 8; })\n    .attr(\'fill\', function(d){ return d.t===\'entity\' ? \'#00c17c18\' : \'#5599ff18\'; })\n    .attr(\'stroke\', function(d){ return d.t===\'entity\' ? \'#00c17c\' : \'#5599ff\'; })\n    .attr(\'stroke-width\', 1.5);\n  node.append(\'text\')\n    .text(function(d){ return d.id.length>12 ? d.id.slice(0,12)+\'...\' : d.id; })\n    .attr(\'fill\', function(d){ return d.t===\'entity\' ? \'#00c17c\' : \'#7aabcc\'; })\n    .attr(\'font-size\',9).attr(\'font-family\',\'JetBrains Mono,monospace\')\n    .attr(\'text-anchor\',\'middle\').attr(\'dy\',\'2.6em\');\n  sim.on(\'tick\', function(){\n    link.attr(\'x1\',function(d){ return d.source.x; }).attr(\'y1\',function(d){ return d.source.y; })\n        .attr(\'x2\',function(d){ return d.target.x; }).attr(\'y2\',function(d){ return d.target.y; });\n    llbl.attr(\'x\',function(d){ return (d.source.x+d.target.x)/2; }).attr(\'y\',function(d){ return (d.source.y+d.target.y)/2; });\n    node.attr(\'transform\', function(d){ return \'translate(\'+d.x+\',\'+d.y+\')\'; });\n  });\n}\nfunction gzi(){ d3.select(\'#gsvg\').transition().call(zb.scaleBy, 1.4); }\nfunction gzo(){ d3.select(\'#gsvg\').transition().call(zb.scaleBy, 0.7); }\nfunction gzr(){ d3.select(\'#gsvg\').transition().call(zb.transform, d3.zoomIdentity); }\n\n// ── stats ─────────────────────────────────────────────────────────────────────\nfunction renderStats() {\n  var s = D.stats.totals;\n  var rows = [\n    {l:\'Total Memories\',v:s.memories||0},{l:\'Entities\',v:s.entities||0},\n    {l:\'Entity Edges\',v:s.entity_edges||0},{l:\'Memory Edges\',v:s.memory_edges||0},\n    {l:\'Sessions\',v:s.sessions||0},{l:\'Daily Logs\',v:s.daily_logs||0}\n  ];\n  var khtml = \'\';\n  for (var i=0;i<rows.length;i++) khtml += \'<div class="kpi"><div class="knum">\'+rows[i].v+\'</div><div class="klbl">\'+rows[i].l+\'</div></div>\';\n  document.getElementById(\'kpis\').innerHTML = khtml;\n  var bt = D.stats.by_type || [];\n  var mx = 1;\n  for (var i=0;i<bt.length;i++) if(bt[i].count>mx) mx=bt[i].count;\n  var bhtml = \'\';\n  for (var i=0;i<bt.length;i++) bhtml += \'<div class="bi"><div class="bl">\'+bt[i].type+\'</div><div class="btr"><div class="bf" style="width:\'+(bt[i].count/mx*100).toFixed(0)+\'%"></div></div><div class="bc">\'+bt[i].count+\'</div></div>\';\n  document.getElementById(\'barchart\').innerHTML = bhtml;\n  var tr = (D.stats.trend||[]).slice().reverse();\n  var mxT = 1;\n  for (var i=0;i<tr.length;i++) if(tr[i].count>mxT) mxT=tr[i].count;\n  var tbhtml=\'\',tlhtml=\'\';\n  for (var i=0;i<tr.length;i++){\n    tbhtml += \'<div class="tbar" style="height:\'+Math.max(3,(tr[i].count/mxT*95))+\'px" title="\'+tr[i].day+\': \'+tr[i].count+\'"></div>\';\n    tlhtml += \'<div class="tlbl">\'+tr[i].day.slice(5)+\'</div>\';\n  }\n  document.getElementById(\'trbars\').innerHTML = tbhtml;\n  document.getElementById(\'trlbls\').innerHTML = tlhtml;\n}\n\n// ── helpers ───────────────────────────────────────────────────────────────────\nfunction esc(s){ return String(s).replace(/&/g,\'&amp;\').replace(/</g,\'&lt;\').replace(/>/g,\'&gt;\').replace(/"/g,\'&quot;\'); }\nfunction fd(s){ if(!s) return \'\'; try{ var d=new Date(s); return d.toLocaleDateString(\'en-GB\',{day:\'numeric\',month:\'short\'}); }catch(e){return s;} }\nfunction fdf(s){ if(!s) return \'\'; try{ return new Date(s).toLocaleString(\'en-GB\'); }catch(e){return s;} }\nfunction toast(msg,err){\n  var t=document.createElement(\'div\'); t.className=\'toast\'+(err?\' err\':\'\'); t.textContent=msg;\n  document.body.appendChild(t); setTimeout(function(){t.remove();},2100);\n}\n</script>\n</body>\n</html>\n'
_F = _T.replace("__JSONDATA__", _DATA)
_url = "data:text/html;base64," + _b64.b64encode(_F.encode("utf-8")).decode("ascii")
_s = _st

display(_IHTML(f"""
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@800&family=JetBrains+Mono&display=swap" rel="stylesheet"/>
<div style="font-family:Syne,sans-serif;background:#090d12;border:1px solid #1d2b3a;border-radius:10px;padding:16px 22px;display:flex;align-items:center;justify-content:space-between;gap:14px;flex-wrap:wrap">
  <div style="display:flex;align-items:center;gap:10px">
    <span style="font-size:20px;filter:drop-shadow(0 0 8px #00c17c80)">&#x1F99E;</span>
    <div>
      <div style="font-size:13px;font-weight:800;color:#dce8f2">OpenClaw <span style="color:#00c17c">x</span> ArangoDB</div>
      <div style="font-size:9px;color:#3d5068;font-family:JetBrains Mono,monospace;letter-spacing:.1em">DIGITAL BRAIN</div>
    </div>
  </div>
  <div style="display:flex;gap:16px">
    <div style="text-align:center"><div style="font-family:JetBrains Mono,monospace;font-size:19px;color:#00c17c">{_s["memories"]}</div><div style="font-size:9px;color:#3d5068;text-transform:uppercase;letter-spacing:.08em">Memories</div></div>
    <div style="text-align:center"><div style="font-family:JetBrains Mono,monospace;font-size:19px;color:#00c17c">{_s["entities"]}</div><div style="font-size:9px;color:#3d5068;text-transform:uppercase;letter-spacing:.08em">Entities</div></div>
    <div style="text-align:center"><div style="font-family:JetBrains Mono,monospace;font-size:19px;color:#00c17c">{_s["entity_edges"]}</div><div style="font-size:9px;color:#3d5068;text-transform:uppercase;letter-spacing:.08em">Edges</div></div>
    <div style="text-align:center"><div style="font-family:JetBrains Mono,monospace;font-size:19px;color:#00c17c">{_s["sessions"]}</div><div style="font-size:9px;color:#3d5068;text-transform:uppercase;letter-spacing:.08em">Sessions</div></div>
  </div>
  <a href="{_url}" target="_blank" style="padding:11px 20px;background:#00c17c;color:#000;border-radius:6px;font-family:Syne,sans-serif;font-size:11px;font-weight:800;letter-spacing:.06em;text-transform:uppercase;text-decoration:none;box-shadow:0 0 14px #00c17c40;white-space:nowrap">
    Open Brain UI &#8594;
  </a>
</div>"""))
print(f"Ready - {len(_mem)} memories | right-click -> Open Link in New Tab if needed")

Ready - 144 memories | right-click -> Open Link in New Tab if needed
